In [1]:
import numpy as np
import pandas as pd

In [2]:
PATH_RAW_DATA = 'data/binefar_bovine_raw.csv'

# Obtener el conjunto de datos en bruto
df_raw = pd.read_csv(PATH_RAW_DATA, sep=';', decimal='.', header=0)

# Tamaño original del conjunto de datos
n_rows, n_cols = df_raw.shape
print(f'Dimensiones:\n- núm. registros: {n_rows}\n- núm. variables: {n_cols}')

Dimensiones:
- núm. registros: 455
- núm. variables: 117


**Selección de variables**. Se seleccionan solo las variables esenciales para la predicción temporal de precios: el marcador de tiempo y los precios medios de bovino agrupados por peso.

In [3]:
# Se conserva fecha y precios del bovino (machos y hembras)
col_names = ['Fecha'] + [
    col for col in list(df_raw.columns)
    if col.startswith('Machos') or col.startswith('Hembras')
]

# Solo se conservan precios medios (quitar columnas que acaban en + o -)
col_names = [
    col for col in col_names if not (col.endswith('+') or col.endswith('-'))
]

# Nuevo número de variables
df_reduced = df_raw[col_names].copy()
n_rows, n_cols = df_reduced.shape
nom_columns = list(df_reduced.columns)

print(f'Dimensiones:\n- núm. registros: {n_rows}\n- núm. variables: {n_cols}\n')
print(f'Variables: {", ".join(nom_columns[:5])}...')

Dimensiones:
- núm. registros: 455
- núm. variables: 33

Variables: Fecha, MachosMenos280KG_E_=, MachosMenos280KG_U_=, MachosMenos280KG_R_=, MachosMenos280KG_O_=...


**Tipos de datos y valores perdidos**. Se ajustan los tipos de datos: fechas en `datetime` para asegurar el orden cronológico y precios en `float32`. Se eliminan registros con todos los valores nulos y los nulos sueltos se rellenan con el último valor conocido de cada variable.

In [4]:
df_reduced['Fecha'] = pd.to_datetime(
    df_reduced['Fecha'],
    format='%d/%m/%Y %H:%M'
).dt.normalize()

df_reduced = df_reduced.set_index('Fecha').sort_index()
df_reduced = df_reduced.astype('float32')

# Eliminar solo los registros sin ningún precio
df_reduced = df_reduced.dropna(how='all').ffill()

**Espurios**. Se aplica filtro de Hampel para series temporales: para cada punto se toma una ventana de vecinos y se calcula su mediana y MAD (desviación respecto a la mediana). Si el punto se aleja de la mediana más de un umbral proporcional a la MAD, se considera atípico y se sustituye por la mediana de la ventana. **No se han detectado errores de registro**.

In [5]:
k = 3
WINDOW = 2 * k + 1
MIN_REL = 0.02
THRESHOLD_FACTOR = 3.0 # umbral: 3 * MAD

# Mediana móvil y MAD
rolling_median = df_reduced.rolling(WINDOW, center=True, min_periods=1).median()
rolling_mad = df_reduced.rolling(WINDOW, center=True, min_periods=1).apply(
    lambda x: np.median(np.abs(x - np.median(x))), raw=True
)
rolling_std = rolling_mad * 1.4826

# Identificar valores atípicos y reemplazar con la mediana de la ventana
threshold =  np.maximum(THRESHOLD_FACTOR * rolling_std, MIN_REL * rolling_median)
is_outlier = (df_reduced - rolling_median).abs() > threshold
df_cleaned = df_reduced.where(~is_outlier, rolling_median)
print(f'- Total espurios sustituidos: {is_outlier.sum().sum()}')

- Total espurios sustituidos: 0


**Intervalos cronológicos uniformes**. En los modelos que procesan secuencias temporales u ordenadas, la distancia temporal es clave y debe mantenerse constante (p. ej., no es recomendable tener registros separados por una semana y, de repente, incluir uno con un salto de cuatro semanas).

In [6]:
from collections import defaultdict

# Intervalos temporales
t_intervals = (df_cleaned.index.to_series().diff().dt.days).to_list()[1:]

# ¿Cuántas veces se produce cada intervalo?
t_counts = defaultdict(int)
for i in t_intervals:
    t_counts[i] += 1

for dias, count in t_counts.items():
    weeks, days = int(k // 7), int(k % 7)
    if days == 0: print(f'- {weeks} semanas: {count} filas')
    else: print(f'- {weeks} semanas y {days} días: {count} filas')

- 0 semanas y 3 días: 437 filas
- 0 semanas y 3 días: 15 filas
- 0 semanas y 3 días: 1 filas
- 0 semanas y 3 días: 1 filas


Como solución habitual se añaden nuevos registros (semanas intermedias) usando ***forward fill***: rellenar los huecos copiando el último valor conocido hacia delante.

In [7]:
idx_semanal = pd.date_range(
    start=df_cleaned.index.min(),
    end=df_cleaned.index.max(),
    name='Fecha',
    freq='7D'
)

df_dataset = df_cleaned.reindex(idx_semanal).ffill()

# Nuevas dimensiones
n_rows, n_cols = df_dataset.shape
print(f'Dimensiones:\n- núm. registros: {n_rows}\n- núm. variables: {n_cols}\n')

# Comprobaciones
t_intervals = (df_dataset.index.to_series().diff().dt.days).iloc[1:]
print('Comprobaciones:')
print(f'- ¿Todos los intervalos son de una semana?: {(t_intervals == 7).all()}')
print(f'- ¿Hay algún valor perdido o nulo en los datos?: {df_dataset.isna().any().any()}')

Dimensiones:
- núm. registros: 476
- núm. variables: 32

Comprobaciones:
- ¿Todos los intervalos son de una semana?: True
- ¿Hay algún valor perdido o nulo en los datos?: False


Se exporta el nuevo fichero de datos `binefar_bovine.csv`.

In [8]:
PATH_CLEAN_DATA = 'data/binefar_bovine.csv'

df_dataset.to_csv(
    PATH_CLEAN_DATA,
    sep=';',
    date_format='%d/%m/%Y',
    float_format='%.4f',
    index=True
)